# Machine-learning models — forecasting and classification

Every ML model in the project, run on the same pooled country-month macro panel built from `fact_macro_indicator` (BIS, World Bank WDI, FRED). Production code (`analysis/`, `etl/`) contains only the inference statistics and the article tagger. These models are exploratory, so they live here.

**Questions.** (A) How large is next month's policy-rate change (regression)? (B) Will next month bring a hike (classification)? (C) Does news stance predict the next move?

**Protocol.**
- Features: six monthly lags of each indicator's change (policy rate, CPI, unemployment, REER, and the FRED globals when ingested).
- Validation: walk-forward (expanding window) on calendar months, 5 outer folds (Bergmeir & Benítez, 2012).
- Hyperparameters: grid search inside each training window with a 3-fold `TimeSeriesSplit`, so no fold ever sees the future.
- Euro area: counted once (Germany's series), since members share the ECB rate.

In [ ]:
import sys, pathlib, time, warnings
sys.path.insert(0, str(pathlib.Path.cwd().parent))
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from scipy import stats
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.linear_model import RidgeCV, LassoCV, ElasticNetCV, LogisticRegression
from sklearn.svm import SVR, SVC
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, HistGradientBoostingRegressor, HistGradientBoostingClassifier
from sklearn.neural_network import MLPRegressor
from sklearn.naive_bayes import GaussianNB
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, confusion_matrix

from common.db import get_store
from common.countries import EURO_ADOPTION
store = get_store()

RUST, CREAM, TEAL = '#b5651d', '#f7f4ee', '#0d9488'
EARTHTONE_SEQUENTIAL = LinearSegmentedColormap.from_list('earthtone_sequential', [CREAM, RUST])
SEED = 0

## Panel

In [ ]:
macro = store.query_df("SELECT country_code, date_key, indicator_id, value_change FROM fact_macro_indicator WHERE date_key >= '1995-01'")
glob = macro[macro.country_code.isna()].pivot_table(index='date_key', columns='indicator_id', values='value_change', aggfunc='first')
ctry = macro[macro.country_code.notna()].pivot_table(index=['country_code', 'date_key'], columns='indicator_id',
                                                     values='value_change', aggfunc='first').reset_index()
wide = ctry.merge(glob.reset_index(), on='date_key', how='left')

# Euro members share one ECB rate after adoption; keep Germany as the single
# euro-area series so identical rows are not counted 21 times.
member = wide.country_code.isin(EURO_ADOPTION) & (wide.country_code != 'DE')
adopted = wide.apply(lambda r: r.date_key >= EURO_ADOPTION.get(r.country_code, '9999'), axis=1)
wide = wide[~(member & adopted)]

FEATURES = [f for f in ['policy_rate', 'cpi_inflation', 'unemployment_rate', 'reer',
                        'fed_funds_rate_global', 'usd_index', 'treasury_yield_10y'] if f in wide]
N_LAGS = 6
frames = []
for c, g in wide.sort_values('date_key').groupby('country_code'):
    if g.policy_rate.notna().sum() < 60:
        continue
    g = g.set_index('date_key')
    X = pd.concat({f'{f}_l{k}': g[f].shift(k) for f in FEATURES for k in range(N_LAGS)}, axis=1)
    X['y'] = g['policy_rate'].shift(-1)   # next month's policy-rate change (pp)
    X['country_code'] = c
    frames.append(X.reset_index())
fc = pd.concat(frames).dropna().sort_values('date_key').reset_index(drop=True)
fc['hike'] = (fc.y > 0).astype(int)
X_COLS = [c for c in fc if c not in ('date_key', 'country_code', 'y', 'hike')]
months = np.sort(fc.date_key.unique())
OUTER = TimeSeriesSplit(n_splits=5)
INNER = TimeSeriesSplit(n_splits=3)
print(f'{len(fc):,} country-months, {fc.country_code.nunique()} countries, {len(X_COLS)} features, {fc.date_key.min()} to {fc.date_key.max()}')
print(f'next-month hike base rate: {fc.hike.mean():.1%}')

## A. Magnitude: next-month policy-rate change

| Model | Family | Hyperparameters (tuned by inner CV where listed) |
| --- | --- | --- |
| `ridge` | L2-penalized linear (Hoerl & Kennard, 1970) | α ∈ logspace(−3, 3, 13), efficient LOO-CV |
| `lasso` | L1-penalized linear (Tibshirani, 1996) | 100-α path, inner `TimeSeriesSplit` |
| `elastic_net` | L1+L2 (Zou & Hastie, 2005) | l1_ratio ∈ {0.2, 0.5, 0.8} × α path |
| `svr_rbf` | Support-vector regression (Cortes & Vapnik, 1995) | C ∈ {0.1, 1}, ε ∈ {0.01, 0.1}, RBF kernel |
| `knn` | k-nearest neighbours (Cover & Hart, 1967) | k ∈ {10, 50, 200} |
| `random_forest` | Bagged trees (Breiman, 2001) | 200 trees, min leaf 20, 30% features per split |
| `hist_gbm` | Histogram gradient boosting, LightGBM-style (Friedman, 2001; Ke et al., 2017) | learning rate ∈ {0.03, 0.1}, leaves ∈ {7, 15}, L2 ∈ {0, 1}, early stopping |
| `mlp` | Feed-forward neural network (Rumelhart et al., 1986) | 32→16 ReLU, L2 α = 1e-2, Adam, early stopping |

Benchmarks: **naive_zero** (no change, the random-walk forecast for a rate level) and **persistence** (repeat this month's change).

In [ ]:
REGRESSORS = {
    'ridge':         lambda: make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-3, 3, 13))),
    'lasso':         lambda: make_pipeline(StandardScaler(), LassoCV(cv=INNER, max_iter=20000)),
    'elastic_net':   lambda: make_pipeline(StandardScaler(), ElasticNetCV(l1_ratio=[.2, .5, .8], cv=INNER, max_iter=20000)),
    'svr_rbf':       lambda: GridSearchCV(make_pipeline(StandardScaler(), SVR()),
                                          {'svr__C': [0.1, 1], 'svr__epsilon': [0.01, 0.1]}, cv=INNER),
    'knn':           lambda: GridSearchCV(make_pipeline(StandardScaler(), KNeighborsRegressor()),
                                          {'kneighborsregressor__n_neighbors': [10, 50, 200]}, cv=INNER),
    'random_forest': lambda: RandomForestRegressor(n_estimators=200, min_samples_leaf=20, max_features=0.3, n_jobs=-1, random_state=SEED),
    'hist_gbm':      lambda: GridSearchCV(HistGradientBoostingRegressor(early_stopping=True, random_state=SEED),
                                          {'learning_rate': [0.03, 0.1], 'max_leaf_nodes': [7, 15], 'l2_regularization': [0, 1]}, cv=INNER),
    'mlp':           lambda: make_pipeline(StandardScaler(), MLPRegressor(hidden_layer_sizes=(32, 16), alpha=1e-2, early_stopping=True,
                                                                          n_iter_no_change=20, max_iter=2000, random_state=SEED)),
}

errors = {k: [] for k in ['naive_zero', 'persistence', *REGRESSORS]}   # pooled out-of-sample errors
fit_s = dict.fromkeys(REGRESSORS, 0.0)
for tr_idx, te_idx in OUTER.split(months):
    tr, te = fc[fc.date_key.isin(months[tr_idx])], fc[fc.date_key.isin(months[te_idx])]
    y = te.y.to_numpy()
    errors['naive_zero'].append(y)
    errors['persistence'].append(y - te.policy_rate_l0.to_numpy())
    for name, make in REGRESSORS.items():
        t = time.perf_counter()
        pred = make().fit(tr[X_COLS], tr.y).predict(te[X_COLS])
        fit_s[name] += time.perf_counter() - t
        errors[name].append(y - pred)
errors = {k: np.concatenate(v) for k, v in errors.items()}

base = np.sqrt(np.mean(errors['naive_zero'] ** 2))
reg_table = pd.DataFrame({
    'RMSE': {k: np.sqrt(np.mean(e ** 2)) for k, e in errors.items()},
    'MAE': {k: np.mean(np.abs(e)) for k, e in errors.items()},
    'fit seconds (5 folds)': pd.Series(fit_s),
})
reg_table['RMSE vs no-change'] = reg_table.RMSE / base - 1
reg_table.sort_values('RMSE').round(4)

In [ ]:
d = reg_table['RMSE vs no-change'].drop('naive_zero').sort_values()
fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.barh(d.index, d.values * 100, color=[TEAL if v < 0 else RUST for v in d])
ax.axvline(0, color='#7a6c5d', linewidth=0.8)
ax.set_xlabel('Walk-forward RMSE vs. no-change forecast (%, teal = better)')
ax.set_title('Next-month policy-rate change: magnitude forecast')
fig.tight_layout()

### Is any improvement real? Diebold–Mariano test

The Diebold–Mariano (1995) test asks whether a model's squared-error loss is significantly different from the no-change benchmark's, on the pooled out-of-sample errors. A negative statistic with p < 0.05 would be a real improvement.

In [ ]:
def diebold_mariano(e_model, e_bench):
    """One-step-ahead DM test on squared-error loss; returns (DM statistic, two-sided p).
    Negative DM = the model has lower loss than the benchmark."""
    d = e_model ** 2 - e_bench ** 2
    dm = d.mean() / np.sqrt(d.var(ddof=1) / len(d))
    return dm, 2 * stats.norm.sf(abs(dm))

dm_table = pd.DataFrame(
    {name: diebold_mariano(errors[name], errors['naive_zero']) for name in REGRESSORS},
    index=['DM statistic', 'p-value']).T.sort_values('DM statistic')
dm_table.round(4)

## B. Direction: will next month bring a hike?

The rate-change *magnitude* is heavy-tailed (emerging-market moves of several percentage points) and mostly zero, so it is hard to forecast. The *direction* is the more useful policy signal. Class weights rebalance the ~10% hike base rate. The metrics are ROC-AUC (threshold-free) and balanced accuracy at 0.5.

| Model | Hyperparameters |
| --- | --- |
| `logistic` (Cox, 1958) | L2, C ∈ {0.01, 0.1, 1} by inner CV on ROC-AUC, balanced class weights |
| `svc_rbf` | C = 1, RBF kernel, balanced, Platt-scaled probabilities |
| `knn` | k = 50 |
| `naive_bayes` (Hand & Yu, 2001) | Gaussian likelihoods |
| `random_forest` | 300 trees, min leaf 20, 30% features, balanced |
| `hist_gbm` | learning rate ∈ {0.03, 0.1}, leaves ∈ {7, 15} by inner CV, balanced |

In [ ]:
CLASSIFIERS = {
    'logistic':      lambda: GridSearchCV(make_pipeline(StandardScaler(), LogisticRegression(class_weight='balanced', max_iter=2000)),
                                          {'logisticregression__C': [0.01, 0.1, 1]}, cv=INNER, scoring='roc_auc'),
    'svc_rbf':       lambda: make_pipeline(StandardScaler(), SVC(C=1, class_weight='balanced', probability=True, random_state=SEED)),
    'knn':           lambda: make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=50)),
    'naive_bayes':   lambda: make_pipeline(StandardScaler(), GaussianNB()),
    'random_forest': lambda: RandomForestClassifier(n_estimators=300, min_samples_leaf=20, max_features=0.3,
                                                    class_weight='balanced', n_jobs=-1, random_state=SEED),
    'hist_gbm':      lambda: GridSearchCV(HistGradientBoostingClassifier(class_weight='balanced', early_stopping=True, random_state=SEED),
                                          {'learning_rate': [0.03, 0.1], 'max_leaf_nodes': [7, 15]}, cv=INNER, scoring='roc_auc'),
}

scores = {k: {'ROC-AUC': [], 'balanced accuracy': []} for k in ['persistence', *CLASSIFIERS]}
fit_s = dict.fromkeys(CLASSIFIERS, 0.0)
last_fold = {}
for tr_idx, te_idx in OUTER.split(months):
    tr, te = fc[fc.date_key.isin(months[tr_idx])], fc[fc.date_key.isin(months[te_idx])]
    persist = (te.policy_rate_l0 > 0).astype(int)
    scores['persistence']['ROC-AUC'].append(roc_auc_score(te.hike, persist))
    scores['persistence']['balanced accuracy'].append(balanced_accuracy_score(te.hike, persist))
    for name, make in CLASSIFIERS.items():
        t = time.perf_counter()
        model = make().fit(tr[X_COLS], tr.hike)
        fit_s[name] += time.perf_counter() - t
        prob = model.predict_proba(te[X_COLS])[:, 1]
        scores[name]['ROC-AUC'].append(roc_auc_score(te.hike, prob))
        scores[name]['balanced accuracy'].append(balanced_accuracy_score(te.hike, prob > 0.5))
        last_fold[name] = (model, te)

clf_table = pd.DataFrame({k: {m: np.mean(v) for m, v in s.items()} for k, s in scores.items()}).T
clf_table['fit seconds (5 folds)'] = pd.Series(fit_s)
clf_table.sort_values('ROC-AUC', ascending=False).round(3)

In [ ]:
best = clf_table['ROC-AUC'].drop('persistence').idxmax()
model, te = last_fold[best]
cm = confusion_matrix(te.hike, model.predict_proba(te[X_COLS])[:, 1] > 0.5)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), gridspec_kw={'width_ratios': [1.4, 1]})
d = clf_table['ROC-AUC'].sort_values()
axes[0].barh(d.index, d.values, color=[RUST if k == best else '#c9bfae' for k in d.index])
axes[0].axvline(0.5, color='#7a6c5d', linewidth=0.8, linestyle='--')
axes[0].set_xlim(0.4, 1)
axes[0].set_xlabel('Walk-forward ROC-AUC (0.5 = chance)')
axes[0].set_title('Next-month rate hike: direction')
im = axes[1].imshow(cm, cmap=EARTHTONE_SEQUENTIAL)
for i in range(2):
    for j in range(2):
        axes[1].text(j, i, f'{cm[i, j]:,}', ha='center', va='center', color='white' if cm[i, j] > cm.max() / 2 else '#2b2118')
axes[1].set_xticks([0, 1], ['no hike', 'hike']); axes[1].set_yticks([0, 1], ['no hike', 'hike'])
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('Actual')
axes[1].set_title(f'{best}, last fold')
fig.tight_layout()

In [ ]:
rf_model, te = last_fold['random_forest']
imp = permutation_importance(rf_model, te[X_COLS], te.hike, scoring='roc_auc', n_repeats=5, random_state=SEED, n_jobs=-1)
by_indicator = pd.Series(imp.importances_mean, index=X_COLS).groupby(lambda c: c.rsplit('_l', 1)[0]).sum().sort_values()
fig, ax = plt.subplots(figsize=(6, 3))
ax.barh(by_indicator.index, by_indicator.values, color=RUST)
ax.set_xlabel('Drop in ROC-AUC when permuted (summed over 6 lags)')
ax.set_title('Random forest: which indicators carry the hike signal')
fig.tight_layout()

## C. News stance → next move (sentiment classifier)

Uses `fact_article.policy_stance` (rule-tagged for every article, LLM-refined as quota allows): the share of a country-month's monetary-policy articles that read contractionary or expansionary, plus mean sentiment. The label is whether the actual policy rate rose the following month. The split is chronological (last 30% of months held out), never shuffled.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report

# One row per country/month: share of that month's monetary_policy articles
# tagged contractionary vs expansionary, plus the label — did the ACTUAL
# policy rate rise next month (contractionary outcome) or not.
stance_df = store.query_df('''
    SELECT bc.country_code, substr(a.date_key, 1, 7) AS date_key,  -- article days -> macro months
           AVG(CASE WHEN a.policy_stance = 'contractionary' THEN 1.0 ELSE 0.0 END) AS pct_contractionary,
           AVG(CASE WHEN a.policy_stance = 'expansionary' THEN 1.0 ELSE 0.0 END) AS pct_expansionary,
           AVG(a.sentiment_score) AS avg_sentiment,
           COUNT(*) AS n_articles
    FROM fact_article a
    JOIN bridge_article_country bc ON a.article_id = bc.article_id
    WHERE a.article_category = 'monetary_policy'
    GROUP BY 1, 2
''')
print(f'{len(stance_df)} country/month rows with tagged monetary-policy sentiment')
stance_df.head()

### Label: did the policy rate actually rise the following month

Joined from `fact_macro_indicator`'s `policy_rate` series — the ground truth this model is trying to predict, completely independent of the sentiment features above.

In [13]:
rate_df = store.query_df('''
    SELECT country_code, date_key, value_change AS rate_change
    FROM fact_macro_indicator WHERE indicator_id = 'policy_rate'
''')
rate_df['date_key'] = pd.to_datetime(rate_df['date_key'])
rate_df['next_month'] = (rate_df['date_key'] + pd.DateOffset(months=-1)).dt.strftime('%Y-%m')
rate_df['contractionary_actual'] = (rate_df['rate_change'] > 0).astype(int)
labels = rate_df[['country_code', 'next_month', 'contractionary_actual']].rename(columns={'next_month': 'date_key'})

train_df = stance_df.merge(labels, on=['country_code', 'date_key'])
print(f'{len(train_df)} rows with both sentiment features and a real next-month label')
train_df.head()

0 rows with both sentiment features and a real next-month label


,country_code,date_key,pct_contractionary,pct_expansionary,avg_sentiment,n_articles,contractionary_actual


### Train + evaluate

**Data-volume note:** every ingested article now carries a rule-tagged `policy_stance`, so this classifier no longer waits on LLM quota. It still needs at least 20 labelled country-months: monetary-policy articles have to overlap months with a policy-rate observation for the same country. Re-run after more feed history accumulates.

In [ ]:
MIN_TRAIN_ROWS = 20

if len(train_df) < MIN_TRAIN_ROWS:
    print(f'Only {len(train_df)} labeled rows (need >= {MIN_TRAIN_ROWS}) — skipping model fit.')
    print('Needs more months where tagged policy articles overlap an observed next-month rate change.')
else:
    train_df = train_df.sort_values('date_key')
    X = train_df[['pct_contractionary', 'pct_expansionary', 'avg_sentiment']]
    y = train_df['contractionary_actual']
    # Chronological hold-out (last 30% of months), not a shuffled split — a shuffled
    # split lets the model see months after the ones it is scored on.
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, shuffle=False)
    # Rate hikes are the minority class; reweight so the model can't win by always predicting 'no hike'.
    clf = LogisticRegression(class_weight='balanced').fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    print(f'Accuracy: {accuracy_score(y_test, y_pred):.2f}')
    print(classification_report(y_test, y_pred, target_names=['not contractionary', 'contractionary']))

In [ ]:
if len(train_df) >= MIN_TRAIN_ROWS:
    cm = confusion_matrix(y_test, y_pred)
    fig, ax = plt.subplots(figsize=(4, 4))
    im = ax.imshow(cm, cmap=EARTHTONE_SEQUENTIAL)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                    color='white' if cm[i, j] > cm.max() / 2 else '#2b2118', fontsize=14)
    ax.set_xticks([0, 1]); ax.set_xticklabels(['not contractionary', 'contractionary'])
    ax.set_yticks([0, 1]); ax.set_yticklabels(['not contractionary', 'contractionary'])
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    ax.set_title('Next-period policy direction — confusion matrix')
    fig.colorbar(im, ax=ax, label='count')
    fig.tight_layout()
else:
    print('No confusion matrix yet — see data-volume note above.')

## D. Production forecaster — `analysis/forecast.py`

The random forest from Part B runs monthly in the Cloud Run analysis job and feeds the UI's *Hike probability* map layer and panel section. It uses the same features and hyperparameters, with one addition: **publication lags**. BIS effective exchange rates trail policy rates by a month, so REER enters one month later, and every feature is limited to data published at forecast time. The run fits on all labelled history and predicts the month after each country's latest data. Euro members receive the euro-area (Germany) forecast. Countries whose latest data is more than 3 months old are skipped.

In [ ]:
from analysis.forecast import run_forecast, PUBLICATION_LAG
print('publication lags:', PUBLICATION_LAG)
prod = run_forecast(write=False)
print(f"{len(prod)} countries | target {prod.target_month.mode()[0]} | walk-forward ROC-AUC {prod.cv_roc_auc.iloc[0]:.3f} | base rate {prod.base_rate.iloc[0]:.1%}")
prod.sort_values('hike_probability', ascending=False)[['country_code', 'as_of_month', 'target_month', 'hike_probability']].head(15)

## Which algorithms fit this problem?

| Algorithm | Applicable? | Evidence here / reason |
| --- | --- | --- |
| Random forest, gradient-boosted trees (XGBoost/LightGBM family) | **Yes — best** | Strongest direction classifiers. They capture non-linear rate-cycle regimes and are fast. `HistGradientBoosting` is the LightGBM algorithm in scikit-learn, so neither extra package is needed |
| Logistic regression | Yes, as an interpretable baseline | Beats persistence, below trees; coefficients are directly reportable |
| Ridge / Lasso / Elastic net | Yes for inference, weak for forecasting | Lasso is used for joint control selection in notebook 02; heavy-tailed targets hurt squared-loss fits |
| SVM (SVR / SVC) | Marginal | O(n²)–O(n³) training is the slowest here and never beats trees; would need linear kernels or subsampling to scale |
| k-NN | No | 42 correlated, non-stationary features defeat distance metrics; near chance |
| Naive Bayes | No for macro, yes for text | Conditional independence fails for lagged macro series, but NB is a classic fast text classifier and could be trained on LLM-labelled articles to replace rules at zero cost |
| MLP / LSTM / CNN (deep learning) | Not at this sample size | MLP underperforms the no-change forecast. Recurrent nets (Hochreiter & Schmidhuber, 1997) need far longer or higher-frequency series |
| Reinforcement learning | No | There is no action/reward environment; the project measures and forecasts, it does not trade |
| VAR / SVAR, local projections | Yes — recommended next | The standard identification tools in this literature (Schischke & Rathgeber, 2025); they answer the causal question correlations cannot |

## References

APA 7th edition. Data sources are cited with the retrieval year of the data the notebook runs on.

### Data

Bank for International Settlements. (2026). *Central bank policy rates* [Data set]. https://data.bis.org/topics/CBPOL

Bank for International Settlements. (2026). *Effective exchange rates* [Data set]. https://data.bis.org/topics/EER

Federal Reserve Bank of St. Louis. (2026). *FRED, Federal Reserve Economic Data* [Data set]. https://fred.stlouisfed.org/

World Bank. (2026). *Commodity markets ("Pink Sheet") data* [Data set]. https://www.worldbank.org/en/research/commodity-markets

World Bank. (2026). *World development indicators* [Data set]. https://databank.worldbank.org/source/world-development-indicators

### Literature and software

Apel, M., & Blix Grimaldi, M. (2014). How informative are central bank minutes? *Review of Economics, 65*(1), 53–76. https://doi.org/10.1515/roe-2014-0104

Bergmeir, C., & Benítez, J. M. (2012). On the use of cross-validation for time series predictor evaluation. *Information Sciences, 191*, 192–213. https://doi.org/10.1016/j.ins.2011.12.028

Breiman, L. (2001). Random forests. *Machine Learning, 45*(1), 5–32. https://doi.org/10.1023/A:1010933404324

Cortes, C., & Vapnik, V. (1995). Support-vector networks. *Machine Learning, 20*(3), 273–297. https://doi.org/10.1007/BF00994018

Cover, T., & Hart, P. (1967). Nearest neighbor pattern classification. *IEEE Transactions on Information Theory, 13*(1), 21–27. https://doi.org/10.1109/TIT.1967.1053964

Cox, D. R. (1958). The regression analysis of binary sequences. *Journal of the Royal Statistical Society: Series B (Methodological), 20*(2), 215–242. https://doi.org/10.1111/j.2517-6161.1958.tb00292.x

Diebold, F. X., & Mariano, R. S. (1995). Comparing predictive accuracy. *Journal of Business & Economic Statistics, 13*(3), 253–263. https://doi.org/10.1080/07350015.1995.10524599

Friedman, J. H. (2001). Greedy function approximation: A gradient boosting machine. *The Annals of Statistics, 29*(5), 1189–1232. https://doi.org/10.1214/aos/1013203451

Hand, D. J., & Yu, K. (2001). Idiot's Bayes—Not so stupid after all? *International Statistical Review, 69*(3), 385–398. https://doi.org/10.1111/j.1751-5823.2001.tb00465.x

Hochreiter, S., & Schmidhuber, J. (1997). Long short-term memory. *Neural Computation, 9*(8), 1735–1780. https://doi.org/10.1162/neco.1997.9.8.1735

Hoerl, A. E., & Kennard, R. W. (1970). Ridge regression: Biased estimation for nonorthogonal problems. *Technometrics, 12*(1), 55–67. https://doi.org/10.1080/00401706.1970.10488634

Ke, G., Meng, Q., Finley, T., Wang, T., Chen, W., Ma, W., Ye, Q., & Liu, T.-Y. (2017). LightGBM: A highly efficient gradient boosting decision tree. *Advances in Neural Information Processing Systems, 30*, 3146–3154.

Loughran, T., & McDonald, B. (2011). When is a liability not a liability? Textual analysis, dictionaries, and 10-Ks. *The Journal of Finance, 66*(1), 35–65. https://doi.org/10.1111/j.1540-6261.2010.01625.x

Pedregosa, F., Varoquaux, G., Gramfort, A., Michel, V., Thirion, B., Grisel, O., Blondel, M., Prettenhofer, P., Weiss, R., Dubourg, V., Vanderplas, J., Passos, A., Cournapeau, D., Brucher, M., Perrot, M., & Duchesnay, É. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research, 12*, 2825–2830.

Rumelhart, D. E., Hinton, G. E., & Williams, R. J. (1986). Learning representations by back-propagating errors. *Nature, 323*(6088), 533–536. https://doi.org/10.1038/323533a0

Schischke, A., & Rathgeber, A. (2025). Commodities and monetary policy—The role of interest rates revisited. *Journal of International Money and Finance, 158*, Article 103416. https://doi.org/10.1016/j.jimonfin.2025.103416

Tibshirani, R. (1996). Regression shrinkage and selection via the lasso. *Journal of the Royal Statistical Society: Series B (Methodological), 58*(1), 267–288. https://doi.org/10.1111/j.2517-6161.1996.tb02080.x

Zou, H., & Hastie, T. (2005). Regularization and variable selection via the elastic net. *Journal of the Royal Statistical Society: Series B (Statistical Methodology), 67*(2), 301–320. https://doi.org/10.1111/j.1467-9868.2005.00503.x